## 1. Initialize Project Environment
Import libraries for data loading, preprocessing, and variance filtering.

In [ ]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")


def locate_repo_root() -> Path:
    """Find the repository root by looking for data folder."""
    here = Path().resolve()
    for base in [here, *here.parents]:
        if (base / "data").exists():
            return base
    raise FileNotFoundError("Could not locate repository root")


REPO_ROOT = locate_repo_root()
DATA_ROOT = REPO_ROOT / "data/work/AndreiCod/lab06"
EXPORT_DIR = REPO_ROOT / "labs/07_network_viz/assignments/artifacts"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

logging.info("Repo root: %s", REPO_ROOT)
logging.info("Data root: %s", DATA_ROOT)
logging.info("Export dir: %s", EXPORT_DIR)
assert (DATA_ROOT / "expression_matrix.csv").exists(), "Expression matrix not found"

## 2. Define Configuration Parameters
Centralize preprocessing options: variance threshold, output paths, and transformation settings.

In [ ]:
@dataclass
class PreprocessConfig:
    handle: str
    expression_csv: Path
    variance_threshold: float = 0.1
    export_dir: Path = None
    apply_log2: bool = True

    def __post_init__(self):
        if self.export_dir is None:
            self.export_dir = EXPORT_DIR

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["expression_csv"] = str(info["expression_csv"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = PreprocessConfig(
    handle="AndreiCod",
    expression_csv=DATA_ROOT / "expression_matrix.csv",
    variance_threshold=0.1,
)
CONFIG.describe()

## 3. Implement Core Functionality
Load the expression matrix, apply log2(x+1), filter low-variance genes, and export the preprocessed data.

In [ ]:
def load_expression_matrix(path: Path) -> pd.DataFrame:
    """Load expression matrix with genes as rows, samples as columns."""
    df = pd.read_csv(path, index_col=0)
    logging.info("Loaded expression matrix: %d genes × %d samples", df.shape[0], df.shape[1])
    return df


def apply_log2_transform(df: pd.DataFrame) -> pd.DataFrame:
    """Apply log2(x+1) transformation to expression values."""
    transformed = np.log2(df + 1)
    logging.info("Applied log2(x+1) transformation")
    return transformed


def filter_low_variance(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """Filter genes with variance below threshold."""
    variances = df.var(axis=1)
    mask = variances >= threshold
    filtered = df.loc[mask]
    logging.info(
        "Filtered genes: %d → %d (removed %d with variance < %.2f)",
        len(df), len(filtered), len(df) - len(filtered), threshold
    )
    return filtered


# Load raw expression data
expr_raw = load_expression_matrix(CONFIG.expression_csv)
expr_raw.head()

In [ ]:
# Summary statistics before preprocessing
print("=== Raw Expression Summary ===")
print(f"Shape: {expr_raw.shape}")
print(f"Value range: [{expr_raw.values.min():.2f}, {expr_raw.values.max():.2f}]")
print(f"Mean variance: {expr_raw.var(axis=1).mean():.2f}")

In [ ]:
# Apply log2 transformation
if CONFIG.apply_log2:
    expr_log = apply_log2_transform(expr_raw)
else:
    expr_log = expr_raw.copy()

print("=== After log2(x+1) ===")
print(f"Value range: [{expr_log.values.min():.2f}, {expr_log.values.max():.2f}]")
print(f"Mean variance: {expr_log.var(axis=1).mean():.4f}")

In [ ]:
# Filter low-variance genes
expr_filtered = filter_low_variance(expr_log, CONFIG.variance_threshold)

print("=== After Variance Filtering ===")
print(f"Shape: {expr_filtered.shape}")
print(f"Genes retained: {len(expr_filtered)} / {len(expr_log)} ({100*len(expr_filtered)/len(expr_log):.1f}%)")

In [ ]:
# Variance distribution
variances = expr_log.var(axis=1)
var_stats = {
    "min": variances.min(),
    "max": variances.max(),
    "mean": variances.mean(),
    "median": variances.median(),
    "threshold": CONFIG.variance_threshold,
    "genes_above_threshold": (variances >= CONFIG.variance_threshold).sum(),
}
pd.DataFrame([var_stats])

## 4. Validate with Unit Tests
Ensure preprocessing functions work correctly on synthetic data.

In [ ]:
def test_log2_transform():
    """Test log2(x+1) transformation."""
    test_df = pd.DataFrame({"S1": [0, 1, 3], "S2": [7, 15, 31]}, index=["G1", "G2", "G3"])
    result = apply_log2_transform(test_df)
    assert np.isclose(result.loc["G1", "S1"], 0.0)  # log2(0+1) = 0
    assert np.isclose(result.loc["G1", "S2"], 3.0)  # log2(7+1) = 3
    assert np.isclose(result.loc["G3", "S2"], 5.0)  # log2(31+1) = 5


def test_variance_filter():
    """Test variance filtering."""
    test_df = pd.DataFrame(
        {"S1": [1, 1, 10], "S2": [1, 2, 20], "S3": [1, 3, 30]},
        index=["low_var", "med_var", "high_var"]
    )
    filtered = filter_low_variance(test_df, threshold=1.0)
    assert "low_var" not in filtered.index
    assert "high_var" in filtered.index


test_log2_transform()
test_variance_filter()
logging.info("All preprocessing tests passed.")

## 5. Export Results
Save the preprocessed expression matrix for use in subsequent tasks.

In [ ]:
# Export preprocessed expression matrix
preprocessed_path = CONFIG.export_dir / "task1_preprocessed_expression.csv"
expr_filtered.to_csv(preprocessed_path)
logging.info("[OK] Preprocessed expression matrix saved to: %s", preprocessed_path.resolve())

# Export preprocessing summary
summary = {
    "original_genes": len(expr_raw),
    "original_samples": expr_raw.shape[1],
    "filtered_genes": len(expr_filtered),
    "variance_threshold": CONFIG.variance_threshold,
    "log2_applied": CONFIG.apply_log2,
}
summary_df = pd.DataFrame([summary])
summary_path = CONFIG.export_dir / "task1_preprocessing_summary.csv"
summary_df.to_csv(summary_path, index=False)
logging.info("[OK] Preprocessing summary saved to: %s", summary_path.resolve())

print(f"\n✓ Task 1 complete: {len(expr_filtered)} genes ready for network construction.")